# 13 — Does the 2D module transfer to new stations and new object designs? Leave-one-station-out (version 2: LOSO-002, fusion for Switch / Outlet)

DIAG-001: on the test stations the proposals are there, but the proposal classifier rejects small objects of a **new design** (test switches = cream plates at 1.14 m, p ≤ 0.34; the Gold switches it was trained on are yellow plates). Two validation stations cannot show this, so here the 2D module is evaluated **leave-one-station-out over the 7 Gold stations** (train on 6, test on the 7th, repeat), with feature sets that depend less on colour / appearance:

| set | parts |
|---|---|
| D (current) | detector scores + 3D context (with colour) + DINOv2 colour crop + DINOv2 intensity crop |
| G | detector scores + 3D context **without colour** + DINOv2 **grey-scale** crop + DINOv2 intensity crop |
| I | detector scores + 3D context without colour + DINOv2 intensity crop |
| N | detector scores + 3D context without colour (geometry only: height, size, flatness, protrusion, range, intensity) |
| B | detector scores + 3D context (with colour) |

Scores (pooled over the 7 held-out stations, 2D module alone, every small class kept, min votes 2 / 3, segment cap 1.5 × the largest training object): **point IoU per small class**, small mIoU, **object recall** (≥ 30 % of an object's points labelled correctly) and **false segments** (predicted segments touching no object of that class). Same views (scan-line scanner position) and proposals as notebook 10 version 4. Test stations are not used.

**Version 2 (LOSO-002).** LOSO-001 (version 1): set D transfers best overall (small mIoU 38.10), but every set reaches only ≈ 20 IoU on Switch; D misses the S165 / S178 switch designs, the geometry-only set N finds S165 (recall 3/5) with more false segments. Version 2 keeps D for Damper, Exit sign and Fire terminal and **fuses D and N probabilities for Switch and Outlet**: p = (1 − w)·p_D + w·p_N, w ∈ {0.3, 0.5, 0.7}, votes 2 / 3; D alone and N alone are recomputed as references. Note: w and votes are chosen on the same LOSO folds that are reported — mild selection bias, stated in the log.

**How to run:** GPU T4 x2, Internet on; no input dataset needed; Save & Run All (about 20 min).

In [ ]:
import os, sys, json, time, subprocess, csv, glob
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy import ndimage
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = DEV == "cpu"
CFG = dict(
    run_name="LOSO-002_fusion_switch_outlet",
    train=["S164", "S165", "S168", "S178", "S179"], val=["S163", "S169"], test=["S170", "S171", "S180"],
    b1_dir="/kaggle/input",                    # where the dataset with the EXP-003 outputs is attached (searched recursively)
    feature_sets_A=[], loso_votes=[2, 3], loso_cap=1.5, fusion_w=[0.3, 0.5, 0.7], fusion_classes=[10, 14],
    thr_grid=[None], min_votes_grid=[1, 2, 3], cap_grid=[None, 1.5, 2.0, 3.0], seg_voxel_m=0.02,
    fixed_variants={"hybrid_H1_mv2": ([3, 10, 14], 2, None), "hybrid_H1_mv2_cap2": ([3, 10, 14], 2, 2.0)},   # (classes, min votes, cap factor)
    final_features="D: + DINOv2 intensity (full)",
    small=[3, 5, 6, 10, 14],                                   # Damper, Exit sign, Fire terminal, Outlet, Switch
    phrases=["light switch", "electrical outlet", "power socket", "smoke detector", "exit sign", "fire extinguisher",
             "fire alarm button", "small device on a wall", "ceiling lamp", "sign", "door plate"],
    gdino="IDEA-Research/grounding-dino-base", sam="facebook/sam-vit-base", dino="facebook/dinov2-base",
    box_thr=0.15, text_thr=0.15, max_boxes=60, max_box_frac=0.08,
    views=[(y, p) for p in (-30, 0, 30) for y in range(0, 360, 45)] + [(0, 90)],
    size=1280, fov=90.0, scanner_height_m=1.6, surface_tol_m=0.02,
    sources=["rgb", "int"], merge_m=0.10, max_extent_m=0.8, min_points=10, label_frac=0.3,
    C=0.5, seed=0, dedup_iou=0.7, same_obj_iou=0.5,
)
if SMOKE:
    CFG.update(train=["S165"], val=["S163"], test=["S170"], views=[(0, 0), (180, 0)], size=800, thr_grid=[None], min_votes_grid=[1, 2], cap_grid=[None, 2.0])
CLASS_NAMES = ["Column", "Components", "Covering", "Damper", "Door", "Exit sign", "Fire terminal", "Furniture",
               "Heater", "Lamp", "Outlet", "Railing", "Slab", "Stair", "Switch", "Wall", "Window", "Clutter"]
OFFSET = np.array([900.0, 480.0, 100.0])
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.path.join("/kaggle/working" if ON_KAGGLE else "./results", CFG["run_name"])
RAW = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
os.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
P = len(CFG["phrases"]); TEXT = " . ".join(CFG["phrases"]) + " ."
YC = [-1] + CFG["small"]                                         # classifier classes: background + the five
YNAME = {-1: "background", **{c: CLASS_NAMES[c] for c in CFG["small"]}}
FSETS = {"A: detector scores": ["gd"], "B: + 3D context": ["gd", "ctx"], "C: + DINOv2 colour": ["gd", "ctx", "er"],
         "D: + DINOv2 intensity (full)": ["gd", "ctx", "er", "ei"], "E: full without detector scores": ["ctx", "er", "ei"]}
print("device:", DEV, "| SMOKE TEST (no GPU) - numbers will mean nothing" if SMOKE else "| GPU run")
print(json.dumps({k: v for k, v in CFG.items() if k != "views"}, indent=1), "\nviews:", len(CFG["views"]))

In [ ]:
# ---- helper functions (notebooks 06 and 07, plus label-free floor / scanner position and candidate features) ----
def instances(xyz, lab, c, link=0.03, min_pts=10):
    """Split the points of class c into separate objects: points closer than `link` metres are joined.
    Returns a list of index arrays (into xyz), largest first, objects with fewer than min_pts points dropped."""
    idx = np.flatnonzero(lab == c)
    if len(idx) == 0:
        return []
    p = xyz[idx]
    pairs = cKDTree(p).query_pairs(link, output_type="ndarray")
    n = len(idx)
    if len(pairs):
        g = coo_matrix((np.ones(len(pairs), dtype=np.int8), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
        _, comp = connected_components(g, directed=False)
    else:
        comp = np.arange(n)
    out = [idx[comp == k] for k in np.unique(comp)]
    out = [o for o in out if len(o) >= min_pts]
    return sorted(out, key=len, reverse=True)


def describe(xyz, inst):
    """Size of one object: extent along its own principal axes (m), number of points, median point spacing (m)."""
    p = xyz[inst]
    q = p - p.mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    ext = np.sort(np.ptp(q @ vt.T, axis=0))[::-1]
    sp = float(np.median(cKDTree(p).query(p, k=2)[0][:, 1])) if len(p) > 1 else float("nan")
    return dict(points=int(len(p)), extent_m=[round(float(e), 3) for e in ext], spacing_m=round(sp, 4),
                centre=[round(float(v), 3) for v in p.mean(0)])


def view_frame(xyz, tree, inst, radius=0.30):
    """Camera looking straight at the surface the object sits on.
    n = normal of the surrounding surface (PCA of the neighbourhood without the object), pointing towards the
    side the object sticks out to; u, v = image axes in that surface (v as close to 'up' as possible)."""
    c = xyz[inst].mean(0)
    nb = np.asarray(tree.query_ball_point(c, radius))
    bg = np.setdiff1d(nb, inst, assume_unique=False)
    if len(bg) < 30:
        bg = nb
    q = xyz[bg] - xyz[bg].mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    n = vt[2]
    p0 = xyz[bg].mean(0)
    if (c - p0) @ n < 0:
        n = -n
    up = np.array([0.0, 0.0, 1.0])
    if abs(n @ up) > 0.9:                       # ceiling / floor: any horizontal axis will do
        up = np.array([0.0, 1.0, 0.0])
    u = np.cross(up, n); u /= np.linalg.norm(u)
    v = np.cross(n, u)
    return c, p0, n, u, v


def render_ortho(xyz, rgb, inten, lab, tree, centre, p0, n, u, v, half, px, depth=(-0.06, 0.20)):
    """Orthographic picture of everything within `half` metres of `centre` (in the image plane) and within the
    depth band in front of / behind the surface. For each pixel the point nearest to the viewer is kept.
    Returns rgb [H,W,3] uint8, intensity [H,W] float (nan = empty), label [H,W] int (-1 = empty)."""
    nb = np.asarray(tree.query_ball_point(centre, half * 1.5 + depth[1]))
    p = xyz[nb]
    d = (p - p0) @ n
    a = (p - centre) @ u
    b = (p - centre) @ v
    m = (d > depth[0]) & (d < depth[1]) & (np.abs(a) < half) & (np.abs(b) < half)
    nb, d, a, b = nb[m], d[m], a[m], b[m]
    W = int(np.ceil(2 * half / px))
    col = np.clip(((a + half) / px).astype(int), 0, W - 1)
    row = np.clip(((half - b) / px).astype(int), 0, W - 1)
    key = row * W + col
    order = np.lexsort((-d, key))               # per pixel: nearest to the viewer first
    ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((W * W, 3), 255, np.uint8); I = np.full(W * W, np.nan, np.float32); L = np.full(W * W, -1, np.int16)
    img[key[sel]] = rgb[nb[sel]]; I[key[sel]] = inten[nb[sel]]
    if lab is not None:
        L[key[sel]] = lab[nb[sel]]
    return img.reshape(W, W, 3), I.reshape(W, W), L.reshape(W, W)


def fill_small_holes(img, I, L, it=1):
    """Fill empty pixels from their neighbours (nearest filled pixel within `it` pixels) - display only."""
    empty = np.isnan(I)
    if not empty.any() or empty.all():
        return img, I, L
    dist, (ri, ci) = ndimage.distance_transform_edt(empty, return_indices=True)
    ok = empty & (dist <= it)
    img = img.copy(); I = I.copy(); L = L.copy()
    img[ok] = img[ri[ok], ci[ok]]; I[ok] = I[ri[ok], ci[ok]]; L[ok] = L[ri[ok], ci[ok]]
    return img, I, L


def estimate_scanner_xy(xyz, lab, floor_class=12, cell=0.02):
    """A terrestrial scanner leaves an unscanned disc on the floor under its tripod. Find the largest empty disc
    that is completely surrounded by floor points. Returns (x, y, radius_m, floor_z) or None."""
    f = xyz[lab == floor_class]
    if len(f) < 1000:
        return None
    lo = f[:, :2].min(0)
    g = ((f[:, :2] - lo) / cell).astype(int)
    occ = np.zeros(g.max(0) + 1, bool); occ[g[:, 0], g[:, 1]] = True
    occ = ndimage.binary_closing(occ, iterations=1)
    holes = ndimage.binary_fill_holes(occ) & ~occ
    if not holes.any():
        return None
    lab_h, nh = ndimage.label(holes)
    dist = ndimage.distance_transform_edt(holes)
    i, j = np.unravel_index(np.argmax(dist), dist.shape)
    r = float(dist[i, j] * cell)
    return float(lo[0] + (i + 0.5) * cell), float(lo[1] + (j + 0.5) * cell), r, float(np.median(f[:, 2]))


def render_panorama(xyz, rgb, inten, lab, origin, deg_per_px=0.1, max_range=30.0):
    """Equirectangular 360 x 180 degree picture seen from `origin`; per pixel the nearest point is kept."""
    q = xyz - origin
    r = np.linalg.norm(q, axis=1)
    m = (r > 0.2) & (r < max_range)
    q, r, idx = q[m], r[m], np.flatnonzero(m)
    az = np.degrees(np.arctan2(q[:, 1], q[:, 0]))          # -180..180
    el = np.degrees(np.arcsin(np.clip(q[:, 2] / r, -1, 1)))  # -90..90
    W = int(round(360 / deg_per_px)); H = int(round(180 / deg_per_px))
    col = np.clip(((az + 180) / deg_per_px).astype(int), 0, W - 1)
    row = np.clip(((90 - el) / deg_per_px).astype(int), 0, H - 1)
    key = row.astype(np.int64) * W + col
    order = np.lexsort((r, key)); ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((H * W, 3), 255, np.uint8); I = np.full(H * W, np.nan, np.float32); L = np.full(H * W, -1, np.int16)
    img[key[sel]] = rgb[idx[sel]]; I[key[sel]] = inten[idx[sel]]
    if lab is not None:
        L[key[sel]] = lab[idx[sel]]
    filled = len(sel) / (H * W)
    return img.reshape(H, W, 3), I.reshape(H, W), L.reshape(H, W), filled


def stretch(I, lo=2, hi=98):
    """Intensity -> 0..255 grey with a percentile stretch (empty pixels white)."""
    v = I[~np.isnan(I)]
    if len(v) == 0:
        return np.full(I.shape, 255, np.uint8)
    a, b = np.percentile(v, [lo, hi]); b = max(b, a + 1e-6)
    g = np.clip((I - a) / (b - a), 0, 1) * 255
    g[np.isnan(I)] = 255
    return g.astype(np.uint8)


def view_dirs(yaw_deg, pitch_deg):
    """Camera axes for a view: f = looking direction, r = image right, d = image down (all unit vectors)."""
    y, p = np.radians(yaw_deg), np.radians(pitch_deg)
    f = np.array([np.cos(p) * np.cos(y), np.cos(p) * np.sin(y), np.sin(p)])
    up = np.array([0.0, 0.0, 1.0]) if abs(pitch_deg) < 80 else np.array([np.cos(y), np.sin(y), 0.0])
    r = np.cross(f, up); r /= np.linalg.norm(r)
    d = np.cross(f, r)                                   # points "down" in the picture
    if d @ up > 0: d = -d
    return f, r, d


def render_view(xyz, rgb, inten, origin, yaw, pitch, size=1280, fov_deg=90.0, fill_px=3, leak=0.06):
    """Pinhole picture from `origin`. For every pixel the nearest point is kept; points that shine through gaps in a
    nearer surface are removed (depth much larger than the local minimum), then small holes are filled from the
    nearest filled pixel. Returns rgb [S,S,3] uint8, intensity [S,S] float32 (nan = empty), idx [S,S] int64
    (index of the 3D point shown in the pixel, -1 = empty)."""
    f, r, d = view_dirs(yaw, pitch)
    q = xyz - origin
    z = q @ f
    m = z > 0.15
    zi = z[m]; qi = q[m]; ii = np.flatnonzero(m)
    fpx = (size / 2) / np.tan(np.radians(fov_deg) / 2)
    u = fpx * (qi @ r) / zi + size / 2
    v = fpx * (qi @ d) / zi + size / 2
    ok = (u >= 0) & (u < size) & (v >= 0) & (v < size)
    u, v, zi, ii = u[ok].astype(np.int64), v[ok].astype(np.int64), zi[ok], ii[ok]
    key = v * size + u
    order = np.lexsort((zi, key)); ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    Z = np.full(size * size, np.inf, np.float32); IDX = np.full(size * size, -1, np.int64)
    Z[key[sel]] = zi[sel]; IDX[key[sel]] = ii[sel]
    Z = Z.reshape(size, size); IDX = IDX.reshape(size, size)
    zmin = ndimage.minimum_filter(Z, size=5, mode="nearest")
    bad = np.isfinite(Z) & (Z > zmin * (1 + leak) + 0.05)
    Z[bad] = np.inf; IDX[bad] = -1
    empty = IDX < 0
    if empty.any() and (~empty).any():
        dist, (ri, ci) = ndimage.distance_transform_edt(empty, return_indices=True)
        fill = empty & (dist <= fill_px)
        IDX[fill] = IDX[ri[fill], ci[fill]]; Z[fill] = Z[ri[fill], ci[fill]]
    img = np.full((size, size, 3), 255, np.uint8); I = np.full((size, size), np.nan, np.float32)
    has = IDX >= 0
    img[has] = rgb[IDX[has]]; I[has] = inten[IDX[has]]
    proj = (u, v, zi, ii, Z)              # every point that falls in the picture, and the front depth per pixel
    return img, I, IDX, proj


def mask_to_points_full(mask, proj, tol=0.03):
    """All 3D points that project into the mask and lie on the visible surface (depth within `tol` of the front
    point of their pixel) - not only the one point per pixel that the picture shows."""
    u, v, zi, ii, Z = proj
    inside = mask[v, u]
    zf = Z[v[inside], u[inside]]
    keep = zi[inside] <= zf + tol
    return np.unique(ii[inside][keep])


def mask_to_points_full(mask, proj, tol=0.03):
    """All 3D points that project into the mask and lie on the visible surface (depth within `tol` of the front
    point of their pixel) - not only the one point per pixel that the picture shows."""
    u, v, zi, ii, Z = proj
    inside = mask[v, u]
    zf = Z[v[inside], u[inside]]
    keep = zi[inside] <= zf + tol
    return np.unique(ii[inside][keep])


def phrase_to_class(phrase, prompts):
    """Map a phrase returned by the detector to a class ID by the words of the prompts (longest match wins)."""
    p = phrase.lower(); best, best_len = None, 0
    for c, plist in prompts.items():
        for pr in plist:
            for w in pr.lower().split():
                if len(w) > 3 and w in p and len(w) > best_len:
                    best, best_len = c, len(w)
    return best


def mask_to_points(mask, IDX):
    """3D point indices behind the pixels of a 2D mask."""
    ids = IDX[mask]
    return np.unique(ids[ids >= 0])




def floor_ceiling(z, step=0.02):
    """Floor and ceiling height without labels: the most populated 2 cm height bin in the lowest 30 % of the height
    range (floor) and between 2.0 and 4.5 m above it (ceiling) - floors and ceilings are large horizontal surfaces."""
    lo, hi = float(np.percentile(z, 0.1)), float(np.percentile(z, 99.9))
    edges = lo - step + step * np.arange(int((hi - lo) / step) + 4)      # one spare bin below and two above
    h, _ = np.histogram(z, bins=edges)
    c = 0.5 * (edges[:-1] + edges[1:])
    low = c < lo + 0.3 * (hi - lo)
    floor = float(c[low][np.argmax(h[low])])
    high = (c > floor + 2.0) & (c < floor + 4.5)                    # ceiling: 2.0-4.5 m above the floor (ignores stairwells)
    if not high.any(): high = c > hi - 0.3 * (hi - lo)
    return floor, float(c[high][np.argmax(h[high])])


def scanner_position(xyz, floor_z, height, cell=0.02):
    """Scanner (x, y) from the unscanned disc on the floor under the tripod, found without labels: floor points =
    within 3 cm of the floor height. Falls back to the centre of the floor. Returns (origin, how)."""
    fm = np.abs(xyz[:, 2] - floor_z) < 0.03
    lab = np.where(fm, 12, -1)
    e = estimate_scanner_xy(xyz, lab, 12, cell)
    if e is not None and e[2] >= 0.15:
        return np.array([e[0], e[1], floor_z + height]), f"tripod disc r={e[2]:.2f} m"
    f = xyz[fm]
    return np.array([np.median(f[:, 0]), np.median(f[:, 1]), floor_z + height]), "floor centre (no tripod disc)"


def phrase_index(label, phrases):
    """Index of the proposal phrase a detector label belongs to (exact match, else most shared words)."""
    l = str(label).lower().strip()
    if l in phrases:
        return phrases.index(l)
    lw = set(l.split()); best, bi = 0, None
    for i, p in enumerate(phrases):
        k = len(lw & set(p.split()))
        if k > best: best, bi = k, i
    return bi


def cluster_centroids(C, r):
    """Group points (proposal centroids) closer than r; returns a component id per point."""
    if len(C) == 0:
        return np.zeros(0, int)
    pairs = cKDTree(C).query_pairs(r, output_type="ndarray")
    n = len(C)
    g = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n)) if len(pairs) else coo_matrix((n, n))
    return connected_components(g, directed=False)[1]


def context_features(xyz, rgb, inten, tree, ids, floor_z, ceil_z, origin, radius=0.30):
    """3D context of one candidate (label-free): height above floor / below ceiling, orientation of the surface it
    sits on, size, how far it sticks out, colour and intensity and their contrast to the surroundings, range."""
    p = xyz[ids]; c = p.mean(0)
    nb = np.asarray(tree.query_ball_point(c, radius))
    bg = np.setdiff1d(nb, ids)
    if len(bg) < 30: bg = nb
    q = xyz[bg] - xyz[bg].mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False); n = vt[2]; p0 = xyz[bg].mean(0)
    if (origin - p0) @ n < 0: n = -n
    prot = (p - p0) @ n
    qq = p - c
    ext = np.sort(np.ptp(qq @ np.linalg.svd(qq, full_matrices=False)[2].T, axis=0))[::-1] if len(p) > 3 else np.zeros(3)
    ib, ic = float(inten[bg].mean()), float(inten[ids].mean())
    cb, cc = rgb[bg].astype(float).mean(0), rgb[ids].astype(float).mean(0)
    return np.array([c[2] - floor_z, ceil_z - c[2], abs(n[2]), ext[0], ext[1], ext[2], np.log1p(len(ids)),
                     np.median(prot), np.percentile(prot, 95), ic, ib, ic - ib, *(cc / 255), *((cc - cb) / 255),
                     float(rgb[ids].astype(float).std() / 255), float(np.linalg.norm(c - origin))], np.float32)

CTX_NAMES = ["h_above_floor", "h_below_ceiling", "abs_normal_z", "ext0", "ext1", "ext2", "log_points", "protrusion_med",
             "protrusion_p95", "int_obj", "int_bg", "int_contrast", "r", "g", "b", "dr", "dg", "db", "rgb_std", "range"]


def scan_order_test(xyz, n=200000):
    """Is the file in scanner order? Median distance between consecutive points (m)."""
    d = np.linalg.norm(np.diff(xyz[:n], axis=0), axis=1)
    return float(np.median(d))


def scanner_by_scanlines(xyz, chunk=100, min_xy=0.15, max_res=0.02, iters=3):
    """If points are stored in scan order, every vertical scan line lies in a vertical plane through the scanner:
    intersect the xy lines of many short runs of consecutive points (least squares, outliers removed)."""
    N, C = [], []
    for s in range(0, len(xyz) - chunk, chunk):
        p = xyz[s:s + chunk]
        if np.ptp(p[:, :2], axis=0).max() < min_xy: continue                       # needs radial movement (floor / ceiling)
        if np.linalg.norm(np.diff(p, axis=0), axis=1).max() > 0.5: continue        # run crosses a jump
        q = p[:, :2] - p[:, :2].mean(0)
        u, sv, vt = np.linalg.svd(q, full_matrices=False)
        if sv[1] / max(sv[0], 1e-9) > 0.05 or np.sqrt(sv[1] ** 2 / chunk) > max_res: continue
        d = vt[0]; N.append([-d[1], d[0]]); C.append(p[:, :2].mean(0))
    if len(N) < 20: return None, 0
    N, C = np.array(N), np.array(C); keep = np.ones(len(N), bool)
    for _ in range(iters):
        A = N[keep]; b = (N[keep] * C[keep]).sum(1)
        x = np.linalg.lstsq(A, b, rcond=None)[0]
        res = np.abs((N * (x - C)).sum(1)); keep = res < max(0.05, np.percentile(res[keep], 80))
    return x, dict(lines=int(keep.sum()), of=len(N), median_residual_m=float(np.median(res[keep])))




def scanner_position_v4(xyz, floor_z, height):
    # FEAS-005: the files keep scanner order -> intersect the vertical planes of short runs of consecutive points
    # (matches the tripod disc within 1.6-4.2 cm); falls back to the tripod disc / floor centre otherwise
    if scan_order_test(xyz) < 0.1:
        x, info = scanner_by_scanlines(xyz)
        if x is not None and info["median_residual_m"] < 0.03:
            return np.array([x[0], x[1], floor_z + height]), f"scan lines ({info['lines']} lines)"
    return scanner_position(xyz, floor_z, height)

In [ ]:
# ---- self-test on a synthetic room (label-free floor and scanner position, merging, context features) ----
def _selftest():
    s = 0.01; Pp, Lb = [], []
    def grid(a0, a1, b0, b1):
        A, B = np.meshgrid(np.arange(a0, a1, s), np.arange(b0, b1, s), indexing="ij"); return A.ravel(), B.ravel()
    x, y = grid(0, 4, 0, 4); k = (x - 1.5) ** 2 + (y - 2.0) ** 2 > 0.3 ** 2
    Pp.append(np.c_[x[k], y[k], np.zeros(k.sum())]); Lb.append(np.full(k.sum(), 12))
    x, y = grid(0, 4, 0, 4); Pp.append(np.c_[x, y, np.full(len(x), 3.0)]); Lb.append(np.full(len(x), 2))
    for X in (0.0, 4.0):
        y, z = grid(0, 4, 0, 3); Pp.append(np.c_[np.full(len(y), X), y, z]); Lb.append(np.full(len(y), 15))
    yy, zz = np.meshgrid(np.arange(0.96, 1.04, 0.003), np.arange(1.06, 1.14, 0.003), indexing="ij")
    Pp.append(np.c_[np.full(yy.size, 0.015), yy.ravel(), zz.ravel()]); Lb.append(np.full(yy.size, 14))
    xyz = np.concatenate(Pp) + np.array([10, 20, 5.0]); lab = np.concatenate(Lb)
    rgb = np.full((len(xyz), 3), 200, np.uint8); inten = np.full(len(xyz), 0.3, np.float32); inten[lab == 14] = 0.8
    fz, cz = floor_ceiling(xyz[:, 2]); assert abs(fz - 5.0) < 0.02 and abs(cz - 8.0) < 0.02, (fz, cz)
    o, how = scanner_position(xyz, fz, 1.6); assert abs(o[0] - 11.5) < 0.03 and abs(o[1] - 22.0) < 0.03, (o, how)
    ids = np.flatnonzero(lab == 14); f = context_features(xyz, rgb, inten, cKDTree(xyz), ids, fz, cz, o)
    assert abs(f[0] - 1.10) < 0.02 and f[2] < 0.1 and f[11] > 0.4 and 0.012 < f[7] < 0.018, f[:12]
    assert phrase_index("light switch", CFG["phrases"]) == 0 and phrase_index("lamp", CFG["phrases"]) == 8
    comp = cluster_centroids(np.array([[0, 0, 0], [0.05, 0, 0], [1, 1, 1.0]]), 0.1); assert comp[0] == comp[1] != comp[2]
    print("self-test passed: floor", round(fz, 3), "ceiling", round(cz, 3), "|", how, "| switch height", round(float(f[0]), 3))
_selftest()

In [ ]:
# ---- models: Grounding DINO (proposals), SAM (masks), DINOv2 (crop embeddings) ----
import transformers
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor, AutoImageProcessor, AutoModel
print("transformers", transformers.__version__, "| torch", torch.__version__)
t0 = time.time()
gd_proc = AutoProcessor.from_pretrained(CFG["gdino"]); gd = AutoModelForZeroShotObjectDetection.from_pretrained(CFG["gdino"]).to(DEV).eval()
sam_proc = SamProcessor.from_pretrained(CFG["sam"]); sam = SamModel.from_pretrained(CFG["sam"]).to(DEV).eval()
dn_proc = AutoImageProcessor.from_pretrained(CFG["dino"]); dn = AutoModel.from_pretrained(CFG["dino"]).to(DEV).eval()
print(f"models loaded in {time.time() - t0:.0f} s | prompt: {TEXT}")

def detect(pil):
    # Grounding DINO -> list of (box, score, phrase index); class-agnostic use: every phrase is kept
    inp = gd_proc(images=pil, text=TEXT, return_tensors="pt").to(DEV)
    with torch.no_grad():
        out = gd(**inp)
    kw = dict(text_threshold=CFG["text_thr"], target_sizes=[pil.size[::-1]])
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=CFG["box_thr"], **kw)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=CFG["box_thr"], **kw)[0]
    labels = r.get("text_labels", r.get("labels"))
    W, H = pil.size; dets = []
    for b, s_, l in zip(r["boxes"].cpu().numpy(), r["scores"].cpu().numpy(), labels):
        pi = phrase_index(l, CFG["phrases"])
        area = max(b[2] - b[0], 0) * max(b[3] - b[1], 0)
        if pi is None or area < 16 or area > CFG["max_box_frac"] * W * H: continue
        dets.append((b.astype(float), float(s_), pi))
    dets.sort(key=lambda d: -d[1])
    return dets[:CFG["max_boxes"]]

def segment(pil, boxes):
    # SAM: best of three masks per box, clipped to the box + 10 %
    res = []
    for i0 in range(0, len(boxes), 32):
        bb = boxes[i0:i0 + 32]
        inp = sam_proc(pil, input_boxes=[[list(map(float, b)) for b in bb]], return_tensors="pt").to(DEV)
        with torch.no_grad():
            out = sam(**inp, multimask_output=True)
        masks = sam_proc.image_processor.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0]
        iou = out.iou_scores[0].cpu().numpy()
        for i, b in enumerate(bb):
            m = masks[i, int(iou[i].argmax())].numpy().astype(bool)
            w, h = b[2] - b[0], b[3] - b[1]; clip = np.zeros_like(m)
            clip[int(max(b[1] - 0.1 * h, 0)):int(b[3] + 0.1 * h) + 1, int(max(b[0] - 0.1 * w, 0)):int(b[2] + 0.1 * w) + 1] = True
            m &= clip
            if m.sum() < 4:
                m = np.zeros_like(m); m[int(b[1]):int(b[3]) + 1, int(b[0]):int(b[2]) + 1] = True
            res.append(m)
    return res

def embed(crops):
    # DINOv2 image embedding (pooled CLS token), L2-normalised
    out = []
    for i0 in range(0, len(crops), 32):
        inp = dn_proc(images=crops[i0:i0 + 32], return_tensors="pt").to(DEV)
        with torch.no_grad():
            o = dn(**inp)
        e = o.pooler_output if getattr(o, "pooler_output", None) is not None else o.last_hidden_state[:, 0]
        out.append(F.normalize(e.float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 768), np.float32)

def crop(pil, b, pad=0.25):
    w, h = b[2] - b[0], b[3] - b[1]; p = pad * max(w, h) + 4
    return pil.crop((int(max(b[0] - p, 0)), int(max(b[1] - p, 0)), int(b[2] + p), int(b[3] + p)))

In [ ]:
# ---- per station: views -> proposals with features (labelled stations: also labels and one example per object) ----
def raw_file(sub, s):
    rel = os.path.join(sub, s + ".npy")
    if os.path.isfile(os.path.join(RAW, rel)): return os.path.join(RAW, rel)
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=[sub + "/" + s + ".npy"], local_dir=RAW)
    return os.path.join(RAW, rel)

def box_iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(x1 - x0, 0) * max(y1 - y0, 0)
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0.0

def merge_dets(dets):
    dets = sorted(dets, key=lambda d: -d[1]); kept = []
    for b, sc, pi in dets:
        if all(box_iou(b, k[0]) < CFG["dedup_iou"] for k in kept):
            kept.append([b, sc, np.zeros(P, np.float32)])
    for b, sc, pi in dets:
        for k in kept:
            if box_iou(b, k[0]) >= CFG["same_obj_iou"]: k[2][pi] = max(k[2][pi], sc)
    return kept

def proposal_label(ids, lab, pid, gsize, gcls):
    fr = {c: float((lab[ids] == c).mean()) for c in CFG["small"]}
    cb = max(fr, key=fr.get)
    if fr[cb] >= CFG["label_frac"]: return cb
    k = pid[ids]; k = k[k >= 0]
    if len(k):
        kk, n = np.unique(k, return_counts=True)
        for kq, nq in zip(kk, n):
            c = gcls[kq]
            if nq >= 0.5 * gsize[kq] and fr[c] >= 0.1: return c
    return -1

def process_station(s, sub):
    t0 = time.time()
    a = np.load(raw_file(sub, s))
    xyz = a[:, :3] - OFFSET; rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8)
    inten = a[:, 6].astype(np.float32); lab = a[:, 7].astype(np.int64) if sub == "Gold" else None
    del a
    fz, cz = floor_ceiling(xyz[:, 2]); origin, how = scanner_position_v4(xyz, fz, CFG["scanner_height_m"])
    tree = cKDTree(xyz)
    gobj = [(c, o) for c in CFG["small"] for o in instances(xyz, lab, c, 0.03, CFG["min_points"])] if lab is not None else []
    pid = np.full(len(xyz), -1, np.int64)
    for k, (c, o) in enumerate(gobj): pid[o] = k
    gsize = np.array([len(o) for c, o in gobj]); gcls = [c for c, o in gobj]
    best = {k: [0, None, None, None] for k in range(len(gobj))}
    gd_gt = np.zeros((len(gobj), P), np.float32)
    props = []
    for vi, (yw, pt) in enumerate(CFG["views"]):
        img, I, IDX, proj = render_view(xyz, rgb, inten, origin, yw, pt, size=CFG["size"], fov_deg=CFG["fov"])
        g = stretch(I); pics = {"rgb": Image.fromarray(img), "int": Image.fromarray(np.dstack([g, g, g]))}
        vis_boxes = {}
        if gobj:
            ov = np.where(IDX >= 0, pid[np.maximum(IDX, 0)], -1)
            for k in np.unique(ov[ov >= 0]):
                ys, xs = np.nonzero(ov == k); b = np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1], float)
                vis_boxes[int(k)] = b
                if len(xs) > best[int(k)][0]:
                    best[int(k)] = [len(xs), crop(pics["rgb"], b), crop(pics["int"], b), b]
        raw = [d for src in CFG["sources"] for d in detect(pics[src])]
        if not raw: continue
        for (b, sc, pi) in raw:
            for k, vb in vis_boxes.items():
                if box_iou(b, vb) > 0.1: gd_gt[k, pi] = max(gd_gt[k, pi], sc)
        dets = merge_dets(raw)
        masks = segment(pics["rgb"], [d[0] for d in dets])
        er = embed([crop(pics["rgb"], d[0]) for d in dets]); ei = embed([crop(pics["int"], d[0]) for d in dets])
        eg = embed([crop(pics["rgb"], d[0]).convert("L").convert("RGB") for d in dets])
        for j, ((b, sc, gdv), m) in enumerate(zip(dets, masks)):
            ids = mask_to_points_full(m, proj, CFG["surface_tol_m"])
            if len(ids) < CFG["min_points"] or np.ptp(xyz[ids], axis=0).max() > CFG["max_extent_m"]: continue
            props.append(dict(ids=ids.astype(np.int32), y=proposal_label(ids, lab, pid, gsize, gcls) if lab is not None else -1, view=vi,
                              gd=np.r_[gdv, gdv.max()].astype(np.float32), ctx=context_features(xyz, rgb, inten, tree, ids, fz, cz, origin),
                              er=er[j], ei=ei[j], eg=eg[j]))
    gtc = []
    ks = [k for k in range(len(gobj)) if best[k][1] is not None]
    if ks:
        er = embed([best[k][1] for k in ks]); ei = embed([best[k][2] for k in ks])
        eg = embed([best[k][1].convert("L").convert("RGB") for k in ks])
        for j, k in enumerate(ks):
            c, o = gobj[k]
            gtc.append(dict(y=c, gd=np.r_[gd_gt[k], gd_gt[k].max()].astype(np.float32),
                            ctx=context_features(xyz, rgb, inten, tree, o, fz, cz, origin), er=er[j], ei=ei[j], eg=eg[j], k=k))
    covered = np.zeros(len(xyz), bool)
    for p_ in props: covered[p_["ids"]] = True
    S = dict(n=len(xyz), lab=lab.astype(np.int8) if lab is not None else None, gobj=gobj, props=props, gtc=gtc,
             prop_recall=[float(covered[o].mean()) >= 0.3 for (c, o) in gobj], how=how, floor=fz, ceiling=cz,
             seconds=round(time.time() - t0, 1), xyz=xyz.astype(np.float32))
    npos = {YNAME[c]: sum(p_["y"] == c for p_ in props) for c in CFG["small"]} if lab is not None else "test station (no labels)"
    print(f"{s} ({sub}): {how}; floor {fz:.2f} ceiling {cz:.2f}; {len(props)} proposals; positives {npos}; "
          + (f"objects covered {sum(S['prop_recall'])}/{len(gobj)}; " if gobj else "") + f"{time.time() - t0:.0f} s")
    return S

STN = {}
t_all = time.time()
for s in CFG["train"] + CFG["val"]:
    STN[s] = process_station(s, "Gold")
# colour-free 3D context: drop r, g, b, dr, dg, db, rgb_std (indices 12-18)
for S in STN.values():
    for it in S["props"] + S["gtc"]: it["ctxnc"] = np.delete(it["ctx"], list(range(12, 19))).astype(np.float32)
print(f"all stations in {(time.time() - t_all) / 60:.1f} min")

In [ ]:
# ---- 2D classifier, 3D predictions of B1 (EXP-003), and their combination ----
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA

def features(train, test, parts, k=32):
    Xtr, Xte = [], []
    for p_ in parts:
        A = np.array([it[p_] for it in train], np.float32); B = np.array([it[p_] for it in test], np.float32)
        if p_ in ("er", "ei", "eg"):
            pca = PCA(n_components=min(k, len(A) - 1), random_state=CFG["seed"]).fit(A)
            A, B = pca.transform(A), pca.transform(B)
        Xtr.append(A); Xte.append(B)
    return np.concatenate(Xtr, 1), np.concatenate(Xte, 1)

def train_predict(train_st, test_st, parts):
    # fit on the proposals + per-object examples of train_st; per test station: per-point mean class probabilities
    train = [it for t in train_st for it in STN[t]["props"] + STN[t]["gtc"]]
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=CFG["C"], class_weight="balanced", max_iter=5000, random_state=CFG["seed"]))
    out = {}
    tests = [STN[s]["props"] for s in test_st]
    allt = [it for L in tests for it in L]
    Xtr, Xte = features(train, allt, parts)
    clf.fit(Xtr, np.array([it["y"] for it in train]))
    pr = clf.predict_proba(Xte) if len(allt) else np.zeros((0, len(clf.classes_)))
    i0 = 0
    for s, L in zip(test_st, tests):
        p_s = pr[i0:i0 + len(L)]; i0 += len(L)
        n = STN[s]["n"]; acc = np.zeros((n, len(clf.classes_)), np.float32); cnt = np.zeros(n, np.int16)
        for it, p in zip(L, p_s):
            acc[it["ids"]] += p; cnt[it["ids"]] += 1
        has = cnt > 0
        acc[has] /= cnt[has][:, None]
        out[s] = dict(mean=acc, cnt=cnt, classes=list(clf.classes_))
    return out

def small_override(base, P2, classes_on, thr=None, min_votes=1):
    # 3D prediction, overwritten by the 2D module where it predicts one of classes_on
    cls = np.array(P2["classes"]); best = P2["mean"].argmax(1); bestp = P2["mean"].max(1)
    lab2 = cls[best]
    ok = (P2["cnt"] >= min_votes) & np.isin(lab2, classes_on)
    if thr is not None: ok &= bestp >= thr
    out = base.copy(); out[ok] = lab2[ok]
    return out

def object_caps(stations, factor):
    # cap per small class = factor x the largest labelled object of that class in `stations` (points); None = no cap
    if factor is None: return {}
    caps = {}
    for t in stations:
        for c, o in STN[t]["gobj"]: caps[c] = max(caps.get(c, 0), len(o))
    return {c: int(factor * v) for c, v in caps.items()}

def segments(xyz, idx, vox):
    # connected segments of the points idx: 2 cm voxels, 26-neighbourhood; returns a segment id per point of idx
    v = np.floor(xyz[idx] / vox).astype(np.int64)
    uv, inv = np.unique(v, axis=0, return_inverse=True); inv = inv.ravel()
    pairs = cKDTree(uv).query_pairs(1.75, output_type="ndarray")
    if len(pairs):
        g = coo_matrix((np.ones(len(pairs), np.int8), (pairs[:, 0], pairs[:, 1])), shape=(len(uv), len(uv)))
        comp = connected_components(g, directed=False)[1]
    else:
        comp = np.arange(len(uv))
    return comp[inv]

def cap_segments(base, out, xyz, caps, vox, stats=None):
    # points changed by the 2D module to class c, in segments larger than caps[c], go back to the 3D label
    out = out.copy()
    for c, cap in caps.items():
        idx = np.flatnonzero((out == c) & (base != c))
        if len(idx) == 0: continue
        seg = segments(xyz, idx, vox); n = np.bincount(seg)
        big = n[seg] > cap
        out[idx[big]] = base[idx[big]]
        if stats is not None:
            stats[CLASS_NAMES[c]] = dict(cap=cap, changed_points=int(len(idx)), segments=int(len(n)), largest=int(n.max()),
                                         removed_segments=int((n > cap).sum()), removed_points=int(big.sum()))
    return out

def hybrid(base, P2, classes_on, mv, factor, xyz, cap_stations, stats=None):
    out = small_override(base, P2, classes_on, None, mv)
    caps = {c: v for c, v in object_caps(cap_stations, factor).items() if c in classes_on}
    return cap_segments(base, out, xyz, caps, CFG["seg_voxel_m"], stats) if caps else out

def _selftest_cap():
    xyz = np.r_[np.random.RandomState(0).rand(500, 3) * 0.05, np.random.RandomState(1).rand(20000, 3) * [1, 1, 0.01] + [5, 5, 0]]
    base = np.zeros(len(xyz), np.int64); out = np.full(len(xyz), 3)
    r = cap_segments(base, out, xyz, {3: 1000}, 0.02)
    assert (r[:500] == 3).all() and (r[500:] == 0).all()
    print("segment-size filter self-test passed")
_selftest_cap()

def find_b1():
    hits = glob.glob(os.path.join(CFG["b1_dir"], "**", "pred_val_2cm_S163.npy"), recursive=True)
    if not hits:
        print("!" * 100 + "\nThe dataset with the EXP-003 outputs is not attached (Add Input -> exp003-b1-outputs). The 3D part and the "
              "combination are skipped" + (" (fine for the smoke test)." if SMOKE else " - THIS RUN WILL NOT PRODUCE THE MAIN RESULT.") + "\n" + "!" * 100)
        return None
    return os.path.dirname(hits[0])

def voxel_subsample_inv(xyz, size):
    g = np.floor(xyz / size).astype(np.int64)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)
    g -= g.min(0)
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    order = np.lexsort((d2, key)); ks = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = ks[1:] != ks[:-1]
    inv = np.empty(len(order), dtype=np.int64); inv[order] = np.cumsum(first) - 1
    return order[first], inv

def confusion(gt, pred, C=18):
    return np.bincount(gt * C + pred, minlength=C * C).reshape(C, C)

def metrics(cm):
    tp = np.diag(cm).astype(float); gt = cm.sum(1); pr = cm.sum(0); present = gt > 0
    iou = np.where(present, tp / np.maximum(gt + pr - tp, 1), np.nan) * 100
    sm = [c for c in CFG["small"] if present[c]]
    return dict(mIoU=round(float(np.nanmean(iou)), 2), small_mIoU=round(float(np.mean([iou[c] for c in sm])), 2),
                OA=round(float(tp.sum() / cm.sum() * 100), 2), iou={CLASS_NAMES[c]: round(float(iou[c]), 2) for c in range(18) if present[c]})

B1 = None      # the 3D network is not used in this notebook

In [ ]:
# ---- LOSO-002: D for Damper / Exit sign / Fire terminal, fused D + N probabilities for Switch / Outlet ----
LS = {"D": ["gd", "ctx", "er", "ei"], "N": ["gd", "ctxnc"]}
GOLD = CFG["train"] + CFG["val"]
FC = CFG["fusion_classes"]
def evaluate_station(s, pred):
    lab = STN[s]["lab"].astype(np.int64); xyz = STN[s]["xyz"]; r = {}
    for c in CFG["small"]:
        g, p = lab == c, pred == c; tp = int((g & p).sum())
        objs = [o for (cc, o) in STN[s]["gobj"] if cc == c]
        hit = sum((pred[o] == c).mean() >= 0.3 for o in objs)
        fp_seg = 0; idx = np.flatnonzero(p)
        if len(idx):
            seg = segments(xyz, idx, CFG["seg_voxel_m"])
            for k in np.unique(seg):
                if not g[idx[seg == k]].any(): fp_seg += 1
        r[c] = np.array([tp, int(g.sum()), int(p.sum()), hit, len(objs), fp_seg])
    return r

def predict_variant(PD, PN, w, mv, n):
    base = np.full(n, -1)
    if w is None:            # D alone
        return small_override(base, PD, CFG["small"], None, mv)
    if w == "N":             # N alone
        return small_override(base, PN, CFG["small"], None, mv)
    assert PD["classes"] == PN["classes"]
    pd_ = small_override(base, PD, [c for c in CFG["small"] if c not in FC], None, mv)
    PF = dict(mean=(1 - w) * PD["mean"] + w * PN["mean"], cnt=PD["cnt"], classes=PD["classes"])
    pf = small_override(base, PF, FC, None, mv)
    out = pd_.copy(); m = np.isin(pf, FC); out[m] = pf[m]
    return out

variants = [("D alone", None), ("N alone", "N")] + [(f"D + fused(w={w}) for Switch/Outlet", w) for w in CFG["fusion_w"]]
RES = {}; PER = {}
for mv in CFG["loso_votes"]:
    tot = {v: {c: np.zeros(6, int) for c in CFG["small"]} for v, _ in variants}
    for h in GOLD:
        tr = [t for t in GOLD if t != h]
        PD = train_predict(tr, [h], LS["D"])[h]; PN = train_predict(tr, [h], LS["N"])[h]
        caps = object_caps(tr, CFG["loso_cap"])
        for vname, w in variants:
            pred = predict_variant(PD, PN, w, mv, STN[h]["n"])
            pred = cap_segments(np.full(STN[h]["n"], -1), pred, STN[h]["xyz"], caps, CFG["seg_voxel_m"])
            r = evaluate_station(h, pred)
            for c in CFG["small"]:
                tot[vname][c] += r[c]
                PER.setdefault(f"{vname} votes{mv}", {}).setdefault(h, {})[CLASS_NAMES[c]] = r[c].tolist()
    for vname, _ in variants:
        row = {}
        for c in CFG["small"]:
            tp, g, p, hit, n, fps = tot[vname][c]
            row[CLASS_NAMES[c]] = dict(iou=round(100 * tp / max(g + p - tp, 1), 2), recall=f"{hit}/{n}", false_segments=int(fps))
        row["small_mIoU"] = round(float(np.mean([row[CLASS_NAMES[c]]["iou"] for c in CFG["small"]])), 2)
        RES[f"{vname} votes{mv}"] = row
        print(f"{vname:<38} votes {mv}: small mIoU {row['small_mIoU']:6.2f} | " + " | ".join(
            f"{CLASS_NAMES[c][:7]} {row[CLASS_NAMES[c]]['iou']:6.2f} rec {row[CLASS_NAMES[c]]['recall']:>5} fp {row[CLASS_NAMES[c]]['false_segments']:>3}" for c in CFG["small"]))
SUMMARY = {"cfg": {k: v for k, v in CFG.items() if k != "views"}, "smoke": SMOKE, "loso": RES, "per_station": PER}
with open(os.path.join(OUT, "summary.json"), "w") as fh: json.dump(SUMMARY, fh, indent=1)
print()
for key in [k for k in PER if k.endswith("votes3")]:
    ps = PER[key]
    print(key, "| Switch:", {h: v["Switch"] for h, v in ps.items() if v["Switch"][4] or v["Switch"][2]})

## After the run

Record as **LOSO-002** in `docs/experiment_log.md` (version 1 = LOSO-001): per variant the pooled LOSO IoU / recall / false segments per small class. If a fused variant clearly beats D alone on Switch / Outlet without more false segments, it becomes the default of the 2D module; otherwise D stays.